# DeploySafe Reddit Market Research
Searches Reddit for people talking about web app security problems that DeploySafe solves.

In [ ]:
# Install dependencies
!pip install requests pygments -q

In [ ]:
# Clone the repo
!git clone https://github.com/salahar9/yars.git
import sys
sys.path.insert(0, 'yars/src')

In [ ]:
import time
from yars.yars import YARS
from yars.utils import export_to_csv, export_post_details_to_csv

scraper = YARS()

In [ ]:
# --- Configure your search here ---

SUBREDDITS = [
    "netsec", "webdev", "devops", "cybersecurity", "AskNetsec",
    "bugbounty", "programming", "sysadmin", "aws", "docker",
    "kubernetes", "django", "node", "rails", "golang",
]

QUERIES = [
    "web app security vulnerability",
    "got hacked production",
    "security scanner tool",
    "penetration testing web app",
    "OWASP vulnerability found",
]

DAYS_BACK = 180          # how far back to look
MIN_SCORE = 5            # ignore low-upvote posts
MIN_COMMENTS = 2         # ignore posts with no discussion
PAGES = 2                # pages per subreddit per query
WORKERS = 5              # parallel subreddit searches

SINCE_UTC = int(time.time()) - (DAYS_BACK * 24 * 60 * 60)

In [ ]:
# --- Search posts across all subreddits for every query ---

all_posts = []
seen_post_ids = set()

for query in QUERIES:
    print(f"\nSearching posts: '{query}'")
    results = scraper.search_subreddits(
        subreddits=SUBREDDITS,
        query=query,
        limit=25,
        time_filter="year",
        since_utc=SINCE_UTC,
        min_score=MIN_SCORE,
        min_comments=MIN_COMMENTS,
        pages=PAGES,
        workers=WORKERS,
    )
    for r in results:
        if r["post_id"] not in seen_post_ids:
            seen_post_ids.add(r["post_id"])
            r["query"] = query
            all_posts.append(r)

print(f"\nTotal unique posts: {len(all_posts)}")
for p in sorted(all_posts, key=lambda x: x['score'], reverse=True)[:5]:
    print(f"  [{p['score']}] {p['title'][:80]} (r/{p['subreddit']})")

In [ ]:
# --- Search comments across all subreddits for every query ---

all_comments = []
seen_comment_ids = set()

for query in QUERIES:
    print(f"\nSearching comments: '{query}'")
    results = scraper.search_subreddits_comments(
        subreddits=SUBREDDITS,
        query=query,
        limit=25,
        time_filter="year",
        since_utc=SINCE_UTC,
        min_score=3,
        pages=PAGES,
        workers=WORKERS,
    )
    for r in results:
        if r["comment_id"] not in seen_comment_ids:
            seen_comment_ids.add(r["comment_id"])
            r["query"] = query
            all_comments.append(r)

print(f"\nTotal unique comments: {len(all_comments)}")
for c in sorted(all_comments, key=lambda x: x['score'], reverse=True)[:5]:
    print(f"  [{c['score']}] {c['body'][:100]} (r/{c['subreddit']})")

In [ ]:
# --- Scrape full comments from the top 5 highest-scored posts ---

top_posts = sorted(all_posts, key=lambda x: x['score'], reverse=True)[:5]
all_thread_rows = []

for post in top_posts:
    print(f"Scraping thread: {post['title'][:70]}")
    permalink = post["link"].replace("https://www.reddit.com", "")
    details = scraper.scrape_post_details(
        permalink,
        max_comments=30,
        min_comment_score=2,
        max_depth=2,
    )
    if details:
        from yars.utils import _flatten_comments
        rows = _flatten_comments(details["comments"], details["title"], details["body"])
        for row in rows:
            row["post_url"] = post["link"]
            row["post_score"] = post["score"]
            row["subreddit"] = post["subreddit"]
        all_thread_rows.extend(rows)

print(f"\nTotal comment rows: {len(all_thread_rows)}")

In [ ]:
# --- Export all results to CSV and download ---

from google.colab import files

if all_posts:
    export_to_csv(all_posts, "deploysafe_posts.csv")
    files.download("deploysafe_posts.csv")

if all_comments:
    export_to_csv(all_comments, "deploysafe_comments.csv")
    files.download("deploysafe_comments.csv")

if all_thread_rows:
    export_to_csv(all_thread_rows, "deploysafe_threads.csv")
    files.download("deploysafe_threads.csv")

print("Done — check your downloads folder.")